In [1]:
import os
import sqlite3
import pandas as pd
from tqdm import tqdm
from langchain_community.document_loaders import CSVLoader #
from langchain_core.documents import Document #
from langchain_text_splitters import RecursiveCharacterTextSplitter #
from langchain_ollama import OllamaEmbeddings #
from langchain_chroma import Chroma #


# VECTORIZE

## ADDRESS PREP

In [2]:
location_mukim_district_state = pd.read_csv('../output/malaysia-postcodes-location-mukim-district-state.csv',dtype='string')
location_mukim_district_state['address'] = location_mukim_district_state.apply(
    lambda row: f"{row['location']}, {row['mukim']}, {row['postcode']}, {row['district']}, {row['state']}", axis=1
) 

location_mukim_district_state['address'] = location_mukim_district_state['address'].apply(lambda x: x.split(', '))
location_mukim_district_state['address'] = location_mukim_district_state['address'].apply(lambda x: [i for i in x if i != '<NA>'])
location_mukim_district_state['address'] = location_mukim_district_state['address'].apply(lambda x: list(dict.fromkeys(x)))
location_mukim_district_state['address'] = location_mukim_district_state['address'].apply(lambda x: ', '.join(x))
address=location_mukim_district_state[['address']]
address.to_csv('../output/address.csv', index=False)

## ADDRESS EMBEDDING

In [3]:
file_path_csv = '../output/address.csv'

# Create a CSVLoader instance
loader = CSVLoader(file_path=file_path_csv)
loader

### CREATE DOCUMENT

In [5]:
# Load documents from CSV file without column headers
documents = loader.load()
documents

# Remode string "address: " from the documents and maintain metadata
documents = [Document(page_content=doc.page_content.replace('address: ', ''), metadata=doc.metadata) for doc in documents]

# Update metadata state by using the last part of the address
for doc in documents:
    address_parts = doc.page_content.split(', ')
    if address_parts:
        doc.metadata['state'] = address_parts[-1]
    if len(address_parts) == 4:
        doc.metadata['district'] = address_parts[-2]
    else:
        doc.metadata['district'] = address_parts[0]

documents

[Document(metadata={'source': '../output/address.csv', 'row': 0, 'state': 'PERLIS', 'district': 'KANGAR'}, page_content='ABI, 01000, KANGAR, PERLIS'),
 Document(metadata={'source': '../output/address.csv', 'row': 1, 'state': 'PERLIS', 'district': 'ARAU'}, page_content='ARAU, 02600, PERLIS'),
 Document(metadata={'source': '../output/address.csv', 'row': 2, 'state': 'PERLIS', 'district': 'KANGAR'}, page_content='BERSERI, 02400, KANGAR, PERLIS'),
 Document(metadata={'source': '../output/address.csv', 'row': 3, 'state': 'PERLIS', 'district': 'PADANG BESAR'}, page_content='CHUPING, 02500, PADANG BESAR, PERLIS'),
 Document(metadata={'source': '../output/address.csv', 'row': 4, 'state': 'PERLIS', 'district': 'KANGAR'}, page_content='UTAN AJI, 01000, KANGAR, PERLIS'),
 Document(metadata={'source': '../output/address.csv', 'row': 5, 'state': 'PERLIS', 'district': 'ARAU'}, page_content='JEJAWI, 01000, ARAU, PERLIS'),
 Document(metadata={'source': '../output/address.csv', 'row': 6, 'state': 'PERL

In [6]:
documents[0]

Document(metadata={'source': '../output/address.csv', 'row': 0, 'state': 'PERLIS', 'district': 'KANGAR'}, page_content='ABI, 01000, KANGAR, PERLIS')

In [7]:
documents[0].page_content[:1000]  # Display the first 1000 characters of the first document


'ABI, 01000, KANGAR, PERLIS'

In [8]:
print(len(documents))
total_docs  = len(documents)

58394


In [9]:
from sentence_transformers import SentenceTransformer
from langchain.embeddings import HuggingFaceEmbeddings

# Initialize the embedding
oembed = OllamaEmbeddings(base_url="http://localhost:11434", model="llama3.2:latest") # 3072-dim
hfembed = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")  # 384-dim

c:\Users\izardy\AppData\Local\miniconda3\envs\etl\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\izardy\AppData\Local\Temp\ipykernel_34460\2698455389.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  hfembed = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")  # 384-dim


In [10]:
# Define the folder path for Chroma's in-memory storage
persist_directory = "../output/vectorstore"

In [ ]:
'''
for i in tqdm(range(0, len(documents))):
    vectorstore = Chroma.from_documents(
        documents=[documents[i]], 
        embedding=oembed, 
        persist_directory=persist_directory,
        collection_name="base_address"  # Specify the collection name here
    )
print('Data Ingested into Vectorstore')
'''

In [11]:
chunk_size = 100
for i in tqdm(range(0, len(documents), chunk_size)):
    batch = documents[i:i+chunk_size]
    Chroma.from_documents(
        documents=batch,
        embedding=hfembed,
        persist_directory=persist_directory,
        collection_name="base_address"
    )


  0%|          | 0/584 [00:00<?, ?it/s]c:\Users\izardy\AppData\Local\miniconda3\envs\etl\lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
100%|██████████| 584/584 [19:44<00:00,  2.03s/it]
